# Data Extraction, Search and Filtering
---
Semantic search is a search technique that leverages the context in a user's query and in the data instead of only keyword matching to provide meaningful responses.  In a nutshell, queries, sentences and documents are represented as dense vectors, called **embeddings**, in a high-dimensional space.  The closer these embeddings are to one another in this space, the more related they are - they share context.  In a properly configured semantic search engine, a query should return only the most similar embeddings from the space.

Vector databases, such as Milvus DB, are excellent at supporting this task.  They provide the ability to efficiently store embeddings as well as computing the similarity between embeddings.

In this runbook, we'll walk through the process of setting up a pipeline to take in unstructured data, pull out useful labels and then use those labels as filters to improve semantic search in Milvus DB.  The use case is a **clinical notes search case**.  Clinicians produce copious notes about their patients, keeping detailed histories.  These histories can be large and searching through them can be a tedious process.  Semantic search with label filtering can ease this process while helping to surface highly relevant information.

To do so, we'll be using the following software:

1. Milvus Standalone Docker as our vector DB ([link](https://milvus.io/docs/prerequisite-docker.md))
2. LangExtract for label extraction ([link](https://github.com/google/langextract))

This particular tutorial uses OpenAI's `gpt-5-mini` as the extraction model for LangExtract and `text-embedding-3-large` as the embedding model.  In practice, you can use whatever models you prefer for these tasks.  

To install Milvus Standalone, follow the linked instructions for your system.  It's recommended that you create a Python virtualenv to contain the packages this project will use.  Python >= 3.10 will work for all the packages being used.  We will begin by first ensuring the necessary packages are installed and importing them.

In [ ]:
%pip install dotenv langextract openai pymilvus

In [ ]:
import os
import re
from dotenv import load_dotenv
import langextract as lx
from openai import OpenAI

from pymilvus import (
    MilvusClient,
    CollectionSchema,
    FieldSchema,
    DataType,
)

## Model Setup
---
After we've imported everything, we'll set up the OpenAI client.  In this tutorial, the OpenAI API key is stored in a `.env` file.  Feel free to retrieve your own API key by any method you prefer.  We'll also create two small helper methods for handling embeddings.

In [ ]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

EMBED_MODEL: str = "text-embedding-3-large"
EXTRACT_MODEL: str = "gpt-5-mini"

openai_client = OpenAI(api_key=api_key)

def embed_notes(notes: list[str]) -> list[list[float]]:
    """Create embeddings from each of the text entries in a given dataset.

    Args:
        notes (list[str]): List of text entries

    Returns:
        list[list[float]]: List of embeddings for each text entry
    """

    # Coerce to a real list and validate types
    if isinstance(notes, str):
        notes = [notes]  # allow single string
    notes = list(notes)

    # Guardrails: non-empty, all strings
    if not notes:
        raise ValueError("embed_notes: empty input list")
    for i, n in enumerate(notes):
        if not isinstance(n, str):
            raise TypeError(f"embed_notes: item {i} is {type(n)}, expected str")

    response = openai_client.embeddings.create(model=EMBED_MODEL, input=notes)
    emb_sorted = sorted(response.data, key=lambda d: d.index)                  # Extra security: ensure embeddings are returned in same order as input notes
    
    return [d.embedding for d in emb_sorted]

def get_embed_dim() -> int:
    """Get the dimensionality of an embedding.  Needed by MilvusDB to create a Collection.

    Returns:
        int: Dimensionality of an embedding produced by the embedding model
    """
    return len(embed_notes(["probe"])[0])


## Database Setup
---
Since we are using Milvus Standalone, it's fairly straightforward to connect the client when the container is up and running.  Once the Milvus client is created, we are ready to create the database, the collection, the schema and the index.  The set up follows these steps:

1. Create a database if doesn't already exist
2. If the collection already exists, drop it
3. Define the fields in the schema
4. Define the schema
5. Create collection based on the schema
6. Create an index over the embedding field to help speed up searching
7. Load the collection into memory

As we are tackling a clinical notes search case, we will construct our schema with that in mind.
1. `id_field`: INT64 - Primary field, set to auto-increment
2. `note_field`: VARCHAR - Contains the raw text of the note
3. `conditon_field`: VARCHAR - Contains the patient's condition ("asthma", "cancer", "diabetes", "hypertension", "unknown")
4. `age_field`: VARCHAR - The age range a patient falls into ("0-17", "18-35", "36-55", "56+")
5. `severity_field`: INT8 - Severity of a patient's condition (range of [1, 5])
6. `emb_field`: FLOAT_VECTOR - Contains the embedding generated from a given note

For the index, we opt to use `FLAT`.  This type of indexing compares a query embedding to every embedding in the space.  This type of index guarantees the highest accuracy and recall, as every potential match is evaluated.  Because of this comparison method, it's slower and can be computationally expensive for larger datasets.  In our case, though, the dataset is very small and we want to the highest possible accuracy when dealing with clinical data, so this indexing method is well-suited.  We also opt to use the `COSINE` similarity metric; it's the default metric in Milvus and offers good performance.


In [ ]:
# Run Milvus Standalone in local docker
MILVUS_URI: str = "http://localhost:19530"
MILVUS_COLL: str = "clinical_notes"
MILVUS_DB: str = "clinical_db"

milvus_client: MilvusClient = MilvusClient(uri=MILVUS_URI)

# Embedding dimensionality
dim: int  = get_embed_dim()

# Only create the DB if it doesn't already exist
if MILVUS_DB not in milvus_client.list_databases():
    milvus_client.create_database(db_name=MILVUS_DB)
    print(f"Created {MILVUS_DB}")
else:
    print(f"{MILVUS_DB} already exists")

# Start with a clean collection
if MILVUS_COLL in milvus_client.list_collections():
    milvus_client.drop_collection(collection_name=MILVUS_COLL)
    print(f"Dropped older {MILVUS_COLL} collection")

# Ensure the correct DB is being used
milvus_client.using_database(db_name=MILVUS_DB)

# Define schema using FieldSchema and CollectionSchema
id_field: FieldSchema = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True)
note_field: FieldSchema = FieldSchema(name="note", dtype=DataType.VARCHAR, max_length=2048)
condition_field: FieldSchema = FieldSchema(name="condition", dtype=DataType.VARCHAR, max_length=32)
age_field: FieldSchema = FieldSchema(name="age_range", dtype=DataType.VARCHAR, max_length=16)
severity_field: FieldSchema = FieldSchema(name="severity_grade", dtype=DataType.INT8)
emb_field: FieldSchema = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dim)

schema = CollectionSchema(
    fields = [id_field, note_field, condition_field, age_field, severity_field, emb_field],
    description = "Clinical notes with guardrails"
)

# Create the collection
collection = milvus_client.create_collection(
    collection_name = MILVUS_COLL, 
    schema = schema
)
print(f"Created {MILVUS_COLL} collection")

# Create an index on the embedding field
# Use FLAT per Milvus recommendation
index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="FLAT",
    metric_type="COSINE",
)

milvus_client.create_index(
    collection_name = MILVUS_COLL,
    index_params=index_params
)

milvus_client.load_collection(
    collection_name=MILVUS_COLL
)

## Data loading and Searching
---
Here we define three helper methods for loading data; building a search string from provided filters; and searching through a collection with a user query and any user-provided filters.

In [ ]:
def ingest_load(collection_name: str, labeled_notes: list[dict]) -> None:
    """Load data into a given collection

    Args:
        collection_name (str): Collection to be loaded into
        labeled_notes (list[dict]): Original text data with its extracted labels
    """

    notes: list = [ln["note"] for ln in labeled_notes]

    embeds: list = embed_notes(notes=notes)

    entities = []
    for ln, emb in zip(labeled_notes, embeds):
        entities.append({
            "note": ln["note"],
            "condition": ln["labels"]["condition"],
            "age_range": ln["labels"]["age_range"],
            "severity_grade": ln["labels"]["severity_grade"],
            "embedding": emb,
        })

    # Insert the data, seal segments, and load collection into memory
    milvus_client.insert(collection_name = collection_name, data = entities)
    milvus_client.flush(collection_name = collection_name)
    milvus_client.load_collection(collection_name = collection_name)

def build_expr(filters: dict[str, str|int]) -> str:
    """Given a set of user-defined filters, construct a filter expression to be used in a search

    Args:
        filters (dict[str, str | int]): Dictionary of user-defined filters and values

    Returns:
        str: Search expression or None if no filters provided
    """

    if not filters:
        return ""
    
    parts = []
    for k, v in filters.items():
        if k == "condition":
            parts.append(f'condition == "{v}"')
        elif k == "age_range":
            parts.append(f'age_range == "{v}"')
        elif k == "severity_grade":
            parts.append(f"severity_grade >= {int(v)}")

    return " AND ".join(parts)

def search(collection_name: str, query: str, filter: dict[str, str|int] = {}, top_k: int = 5) -> list[dict]:
    """Given a collection, a user query and an optional set of filters, search the collection for query response(s)

    Args:
        collection_name (str): Collection to be searched
        query (str): User query
        filter (dict[str, str | int], optional): Dictionary of user-defined filters and values. Defaults to {}.
        top_k (int, optional): Number of results to return. Defaults to 5.

    Returns:
        list[dict]: List of responses that are most similar to the query
    """

    # Convert the query into an embedding
    embed = embed_notes(notes=[query])[0]

    # Build the search expression
    expr = build_expr(filters = filter)

    results = milvus_client.search(
        collection_name = collection_name,
        data = [embed],
        anns_field = "embedding",
        search_params = {"metric_type": "COSINE"},
        limit = top_k,
        filter = expr,
        output_fields = ["note", "condition", "age_range", "severity_grade"], 
    )

    response = [
        {
            "score": float(hit.distance),
            "text": hit.entity.get("note"),
            "condition": hit.entity.get("condition"),
            "age_range": hit.entity.get("age_range"),
            "severity_grade": hit.entity.get("severity_grade"),
        } for hit in results[0]
    ]

    return response

## Dataset, Labels and Priming Extraction
___
We define here the synthetic dataset to be used in this tutorial, the labels that LangExtract will use, as well as the prompt and examples that will set up LangExtract to carry out the extraction.  The last two points are especially important as LangExtract is a "few-shot" system: you must provide examples of how to apply labels to the dataset to start the extraction process.

To further improve the extraction process, we will leverage LangExtract's optional `attributes` argument in the provided examples.  This argument lets us associate additional metadata and/or relationships to the extraction.  The attributes we use will encode "normalized" values into the extractions (the values expected in CONDITIONS, AGE_RANGES, and SEVERITY_GRADE/SEVERITY_MAP).  This inclusion will reduce the amount of post-processing necessary to be able to apply filtering to semantic search as LangExtract will try to output normalized values.

We can reinforce the normalization with an appropriately-written prompt.  This is especially important for extracting severity values, as they can be integers or strings in the note, but must only be integers when included in the database.  The prompt explicitly calls out: 
1. That words such as "mild" and "severe" should be mapped to values of 1 and 4, respectively.
2. That severity values given as strings should be returned as integers, e.g., "severity 2" should be 2.
3. And, if no severity value or word is present in the note, the severity attribute should be set to 1.

In [ ]:
CONDITIONS: list[str] = ["asthma", "cancer", "diabetes", "hypertension", "unknown"]
AGE_RANGES: list[str] = ["0-17", "18-35", "36-55", "56+"]
DEFAULT_AGE_RANGE: str = "36-55"
SEVERITY_MAP: dict[str, int] = {
    "mild": 1,
    "moderate": 3,
    "severe": 4,
    "critical": 5,
}

NOTES: list[str] = [
        "62-year-old with poorly controlled type 2 diabetes, A1c rising, moderate neuropathy; severity 3. Adjust meds.",
        "Pediatric patient (age 12) presenting with mild asthma exacerbation after exercise; inhaler use increased.",
        "Hypertension follow-up for a 45-year-old; BP remains elevated despite ACE inhibitor; consider dose escalation.",
        "Oncology referral for suspected malignancy; 56-year-old with weight loss and persistent cough; severity 4.",
        "ER visit due to severe asthma attack; 33-year-old required nebulizer; ICU admission considered; critical condition.",
        "Routine check-up: 28-year-old with stable diabetes on metformin; A1c improved; severity 2.",
        "Elderly patient with uncontrolled high blood pressure; 77-year-old; headaches, dizziness; severity 4.",
        "Bronchial asthma in 19-year-old with moderate persistent symptoms; inhaled corticosteroids adjusted.",
        "Follow-up: 53-year-old with cancer in remission; chemotherapy completed; mild neuropathy; severity 2.",
        "General consult for fatigue in a 38-year-old; labs pending; no clear condition yet.",
        "Hypertension (HTN) in 60-year-old; ER visit last week for hypertensive urgency; severity 5.",
        "Type 2 DM with complications in 47-year-old; foot ulcer forming; poorly controlled; severity 4.",
        "Asthma well-controlled in a 24-year-old athlete; rare albuterol use; severity 1.",
        "Possible oncology case: 58-year-old with suspicious mass; oncology review pending; severity 3.",
        "BP borderline in 35-year-old; lifestyle counseling; severity 1.",
        "12-year-old presenting with mild asthma exacerbation.",
        "45-year-old with hypertension, severity grade 4.",
        "33-year-old with severe asthma attack requiring nebulizer, critical condition",
        "Elderly patient (age 77) with hypertension, headaches; severity 4.",
]

prompt_description = """
Extract structured clinical attributes:
1) CONDITION in {diabetes, hypertension, asthma, cancer}, else 'unknown'
2) AGE_RANGE in {0-17, 18-35, 36-55, 56+}
3) SEVERITY_GRADE must have attributes.normalized as an integer in range [1..5].
   - If the note uses words (mild, moderate, severe, critical), set normalized to {1,3,4,5}.
   - If the note gives a digit (e.g., "severity 3"), set normalized to that integer.
   - If no severity is stated, still emit a severity_grade entity with attributes.normalized = 1
Return entities in the order they appear. Use extraction_text as a literal span from the note; put the final canonical value in attributes.normalized.
"""

examples: list = [
    lx.data.ExampleData(
        text="62-year-old with poorly controlled type 2 diabetes; severity 3 neuropathy.",
        extractions=[
            lx.data.Extraction(extraction_class="condition", extraction_text="diabetes", attributes={"normalized": "diabetes"}),
            lx.data.Extraction(extraction_class="age_range", extraction_text="62-year-old", attributes={"normalized": "56+"}),
            lx.data.Extraction(extraction_class="severity_grade", extraction_text="3", attributes={"normalized": 3}),
        ]
    ),
    lx.data.ExampleData(
        text="12-year-old presenting with mild asthma exacerbation.",
        extractions=[
            lx.data.Extraction(extraction_class="condition", extraction_text="asthma", attributes={"normalized": "asthma"}),
            lx.data.Extraction(extraction_class="age_range", extraction_text="12-year-old", attributes={"normalized": "0-17"}),
            lx.data.Extraction(extraction_class="severity_grade", extraction_text="mild", attributes={"normalized": 1}),
        ]
    ),
    lx.data.ExampleData(
        text="33-year-old with severe asthma; ICU evaluation discussed.",
        extractions=[
            lx.data.Extraction("condition", extraction_text="asthma", attributes={"normalized": "asthma"}),
            lx.data.Extraction("age_range", extraction_text="33-year-old", attributes={"normalized": "18-35"}),
            lx.data.Extraction("severity_grade", extraction_text="severe", attributes={"normalized": 4}),
        ],
    ),
    lx.data.ExampleData(
        text="70-year-old with critical COPD exacerbation.",
        extractions=[
            lx.data.Extraction("condition", extraction_text="COPD", attributes={"normalized": "unknown"}),  # Disallowed diseases normalize to unknown
            lx.data.Extraction("age_range", extraction_text="70-year-old", attributes={"normalized": "56+"}),
            lx.data.Extraction("severity_grade", extraction_text="critical", attributes={"normalized": 5}),
        ],
    ),
    lx.data.ExampleData(
        text="Check up for fatigue in patient (age 25); labs pending; no clear condition yet.",
        extractions=[
            lx.data.Extraction("condition", extraction_text="no clear condition", attributes={"normalized": "unknown"}),
            lx.data.Extraction("age_range", extraction_text="(age 25)", attributes={"normalized": "36-55"}),
            lx.data.Extraction("severity_grade", extraction_text="check up", attributes={"normalized": 1}),     # default when absent
        ],
    )
]

## Extraction
---
These helper methods are for the extraction process.  The key method is `normalize_extracts()`.  The data captured by the label extraction needs additional post-processing to be used in filter expressions consumable by Milvus search.  The post-processing covers:

1. In the case where LangExtract finds a condition that isn't in the conditions list, it gets marked as `unknown`.
2. In clinical notes, ages are represented as single numbers, so they need to be placed into the appropriate age ranges.
3. Finally, the severity of a case can be numerical or even a word, e.g., "mild" or "critical," in a note.  Severity can only be a number in the range of [1, 5].

When normalizing extractions, there are 2 significants checks to perform:
1. Check that the "normalized" attribute is present as that is preferred for use; if it isn't, use the extraction_text (which should always be present).
2. Severity still needs to be checked as LangExtract is powered by an LLM: it can't be guaranteed that it will always output normalized integer values.

The CONDITION and AGE_RANGE values are pure strings in well-defined sets, so they need less stringent bounding.  But, as an exercise for the reader, how might you go about ensuring you only get the expected values?

In [ ]:
def extractor(text: str):
    """Wrapper around the call to LangExtract.extract().

    Args:
        text (str): Note from which labels will be extracted

    Returns:
        _type_: The LangExtract extraction object
    """
    result = lx.extract(
        text_or_documents = text,
        prompt_description = prompt_description,
        examples = examples,
        model_id = EXTRACT_MODEL,
        api_key = api_key
    )

    return result

def normalize_extracts(extractions) -> dict[str, str|int]:
    """Normalize extracted labels to be inline with accepted conditions, age ranges and severities.

    Args:
        extractions (Extraction): Extraction for a given note

    Returns:
        dict[str, str|int]: The normalized labels
    """

    # Labels are extraction_class
    # Use getattr() to check that attributes are present and to use the "normalized" response
    # Otherwise, use the extraction_text (which is required in a LangExtract example)
    labels: dict = {}
    for extract in extractions.extractions:
        if getattr(extract, "attributes", None) and "normalized" in extract.attributes:
            labels[extract.extraction_class] = extract.attributes["normalized"]
        else:
            labels[extract.extraction_class] = extract.extraction_text

    # CONDITION
    condition: str = labels.get("condition", "").lower()
    if condition not in CONDITIONS:
        condition: str =  "unknown"

    # AGES
    # There is an implicit assumption that the age will always be numerical
    age_extract_range: str = labels.get("age_range", "")

    # SEVERITY
    # This check is necessary as the LangExtract model can "slip" and end up returning
    # a string for the severity value or an integer outside the specified range
    sev_extract = labels.get("severity_grade", "")
    severity: int

    if isinstance(sev_extract, int):
        severity = max(1, min(5, sev_extract)) # Clamp to [1..5]
    elif isinstance(sev_extract, str):
        s: str = sev_extract.strip().lower()
        if s.isdigit():
            severity = max(1, min(5, int(s))) # Clamp to [1..5]
        else:
            severity = SEVERITY_MAP.get(s, 1) # If a descriptive word is returned
    else:
        severity = 1 # If there is no explicit severity in the note

    return {
        "condition": condition,
        "age_range": age_extract_range,
        "severity_grade": severity
    }

def extract_labels(notes: list[str]) -> list[dict[str, str|dict]]:
    """Wrapper around the extractor and normalize_extracts methods for ease of use

    Args:
        notes (list[str]): Notes to be processed for labels

    Returns:
        list[dict[str, str|dict]]: Dictionary containing the original notes alongside their label extractions
    """
    results: list = []

    for n in notes:
        extractions = extractor(n)
        labels = normalize_extracts(extractions)
        results.append({"note": n, "labels": labels})

    return results

## Querying
---
Here we finally run a couple queries, both with and without filters.  How long this process takes depends on the size of the dataset; the models we're using for extraction and embedding; and any delays brought on by the model vendors (e.g., rate limits) or the network.  

With these queries we can clearly see how adding label filters can help with semantic search.  In both cases, the semantic search yields results dealing with asthma, diabetes and symptom management.  However, in the no-filter case, the responses tend to cover a range of ages and severities.  With filtering, though, we can focus the search to pull out specific data.  This is important for any sort of search: you want specific information to answer your questions.

In [ ]:
labels = extract_labels(NOTES)
ingest_load(collection_name = MILVUS_COLL, labeled_notes = labels)

In [ ]:
query = "shortness of breath after activity; how to manage symptoms?"
query2 = "patient with low blood sugar; treat symptoms"

print("\nQUERY 1\n-------")
print("\nWITHOUT FILTERS:")
for r in search(collection_name = MILVUS_COLL, query = query, top_k = 3):
    print(r)

print("\nWITH FILTERS (condition=asthma, severity>=3, age_range=18-35):")
for r in search(collection_name = MILVUS_COLL, 
                query = query, 
                filter={"condition": "asthma", "severity_grade": 3, "age_range": "18-35"}, 
                top_k = 3):
    print(r)

print("\nQUERY 2\n-------")
print("\nWITHOUT FILTERS:")
for r in search(collection_name = MILVUS_COLL, query = query2, top_k = 3):
    print(r)

print("\nWITH FILTERS (condition=diabetes, severity>=2, age_range=18-35):")
for r in search(collection_name = MILVUS_COLL, 
                query = query2, 
                filter={"condition": "diabetes", "severity_grade": 2, "age_range": "18-35"}, 
                top_k = 3):
    print(r)

## Next Step
---
As an exercise for the reader, try integrating the results of the semantic search into a RAG.  You can find several tutorials on how to do just that in the [Milvus Bootcamp](https://milvus.io/bootcamp).  Good luck and have fun!